# c2_90—Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

It writes to an ROI-owned bucket and will throw a permissions error for anyone else. It is in
the repo because it belongs next to the thing it mirrors, not because students should run it.

## The important design decision

**This notebook does not reimplement the transformations.** It fetches
`c2_01_load_explore.ipynb` by raw URL and executes it once per metro with `METRO` overridden,
exporting the resulting dataframes to Parquet.

That guarantees the snapshot cannot drift from what students actually get. The moment someone
edits the student notebook, this picks the change up on the next run. A hand-maintained copy
of the same logic would be wrong within a week.

## Why it loads BigQuery tables, when all it needs is dataframes

An earlier version of this notebook stripped the `load_table` calls out, on the reasoning that
we are exporting dataframes and therefore do not need the tables. That was wrong, and wrong in
a way worth writing down.

The student notebook's validation section—23 checks—runs its queries **against the loaded
BigQuery tables**, not against the dataframes in memory. With the writes stripped out, those
queries still ran: they simply read whatever happened to be sitting in the maintainer's
`a4i_food` dataset from the last time they ran the student notebook by hand. So the checks
passed for every metro, and what they were checking was one stale copy of Chicago.

A validation suite reading the wrong data is worse than no validation suite, because it is
reassuring. So we now load each metro into a **scratch dataset** (`a4i_food_publish`, not the
maintainer's own `a4i_food`), let the validation run against it for real, and **refuse to
export a metro whose checks did not pass**.

## Why the snapshot exists at all

`c2_01_load_explore.ipynb` pulls live from irs.gov, foodsafety.gov, and Seattle's Socrata API.
Three external dependencies, on a day when 150 people hit them inside the same ten minutes.
If any one is down or rate-limiting, the notebook fails for the entire room at once.

`scripts/load.sh` rebuilds every table from this snapshot instead, so a single upstream outage
costs a team five minutes rather than their afternoon.

## Bucket layout

```
gs://class-demo/a4i-2026/challenge-2-food-equity/<metro>/<table>/*.parquet
```

`a4i-2026` is the generic top level every challenge shares.
The bucket needs `allUsers:objectViewer` so students can read anonymously from their own
projects—`load.sh` does not authenticate against it.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET  = "class-demo"
PREFIX  = "a4i-2026/challenge-2-food-equity"

NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-2-food-equity/main/notebooks/c2_01_load_explore.ipynb")

# Where the per-metro tables land so the student notebook's validation section
# has something real to query. Deliberately NOT "a4i_food": that is the dataset
# a maintainer uses when running the student notebook by hand, and this notebook
# truncates every table in here nine times in a row.
SCRATCH_DATASET = "a4i_food_publish"
LOCATION        = "US"

# Every metro we publish, with the two facts that let us prove afterwards that
# the metro override actually took: the state its organizations must come from,
# and the two-digit state FIPS every one of its census tract ids must start with.
#
# This pairing exists because of a real incident. A refactor of the student
# config cell broke the METRO substitution below, the substitution failed
# silently, and all nine metros published Chicago's data. Everything loaded,
# every row count looked plausible, and the snapshot was wrong. Row counts do
# not tell you whose city you are looking at. These two columns do.
#
# Adding a metro here is the only change needed - load.sh discovers what exists
# by listing the bucket.
METROS = {
    "Chicago":      ("IL", "17"),
    "Dallas":       ("TX", "48"),
    "Seattle":      ("WA", "53"),
    "Philadelphia": ("PA", "42"),
    "Atlanta":      ("GA", "13"),
    "Houston":      ("TX", "48"),
    "Denver":       ("CO", "08"),
    "New York":     ("NY", "36"),
    "Phoenix":      ("AZ", "04"),
}

TABLES = {
    "recipients":         "out",              # variable name in the student notebook
    "surplus_postings":   "surplus",
    "shelf_life":         "shelf_life",
    "tract_demographics": "tracts",
}

import google.auth
credentials, PROJECT_ID = google.auth.default()
print(f"Publishing from project: {PROJECT_ID}")
print(f"Scratch dataset       : {PROJECT_ID}.{SCRATCH_DATASET} ({LOCATION})")
print(f"Metros                : {', '.join(METROS)}")
print(f"Target                : gs://{BUCKET}/{PREFIX}/<metro>/<table>/")

## Fetch the student notebook and extract its code

We take the code cells and drop two kinds we must not execute:

- **`%%bigquery` magics** — `exec` cannot run cell magics at all. These are display cells in the
  student notebook, so nothing downstream depends on them.
- **Appendix B** — the standalone city-viability tester. It is a diagnostic, not part of the
  pipeline, and running it nine times would re-fetch irs.gov nine more times for nothing. It
  carries a marker comment naming this notebook, so the skip survives someone retitling it.

**The `load_table` calls stay.** See the note at the top: stripping them is what let a stale
`a4i_food` dataset stand in for nine metros' worth of validation.

Then we rewrite two lines in the config cell — `METRO`, and `DATASET` so the loads land in the
scratch dataset rather than the maintainer's own. Both rewrites are located **once**, up front,
and test-fired on a probe before the run starts. A rewrite that silently matches nothing is the
failure mode this notebook is most exposed to, so it is checked rather than assumed.

In [ ]:
import re
import requests
import nbformat

SKIP_MARKER = "c2_90_publish_snapshot.ipynb skips this cell"

nb = nbformat.reads(requests.get(NOTEBOOK_URL, timeout=60).text, as_version=4)

sources, skipped = [], []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if src.lstrip().startswith("%%"):           # exec() cannot run cell magics
        skipped.append("magic")
        continue
    if SKIP_MARKER in src:                      # Appendix B and anything else opted out
        skipped.append("marked")
        continue
    sources.append(src)                         # load_table calls included, on purpose

# --- Find the config cell, once, and prove it is unambiguous ----------------
# The student cell reads:
#     DEFAULT_METRO = "Chicago"
#     METRO = DEFAULT_METRO
#     ...
#     DATASET  = "a4i_food"
# We rewrite METRO's line, not DEFAULT_METRO's, so DEFAULT_METRO keeps its
# meaning and the "you are on the default" nudge only fires on the run that
# really is. DATASET is redirected so we never truncate a maintainer's own work.
METRO_LINE   = re.compile(r'(?m)^METRO\s*=\s*.+$')     # not METROS - no '=' after 'METRO'
DATASET_LINE = re.compile(r'(?m)^DATASET\s*=\s*[^\n]+')

config_idx = [i for i, s in enumerate(sources)
              if "DEFAULT_METRO" in s and METRO_LINE.search(s) and DATASET_LINE.search(s)]
if len(config_idx) != 1:
    raise RuntimeError(
        f"Expected exactly one config cell assigning both METRO and DATASET; found "
        f"{len(config_idx)}: {config_idx}. The student notebook's config cell has changed "
        f"shape and the overrides below would not work. Fix this before publishing anything."
    )
CONFIG_IDX = config_idx[0]


def configure(src, metro):
    """Rewrite METRO and DATASET in the config cell. Raises rather than no-ops."""
    src, n = METRO_LINE.subn(lambda _: f'METRO = "{metro}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"METRO substitution matched {n} lines, expected 1")
    src, n = DATASET_LINE.subn(lambda _: f'DATASET = "{SCRATCH_DATASET}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"DATASET substitution matched {n} lines, expected 1")
    return src


# Prove both substitutions work before we spend an hour finding out they do not.
probe = configure(sources[CONFIG_IDX], "Denver")
assert 'METRO = "Denver"' in probe, "metro override did not take"
assert f'DATASET = "{SCRATCH_DATASET}"' in probe, "dataset override did not take"
assert "DEFAULT_METRO" in probe, "DEFAULT_METRO was clobbered"
assert '"a4i_food"' not in probe, "the student dataset name survived the override"

print(f"Code cells in the student notebook : {sum(1 for c in nb.cells if c.cell_type == 'code')}")
print(f"Skipped (magic / marked)           : {skipped.count('magic')} / {skipped.count('marked')}")
print(f"Cells we will execute              : {len(sources)}")
print(f"Config cell                        : index {CONFIG_IDX}, both overrides verified")

## Build, validate, then export—one metro at a time

Each metro runs in its own namespace so a failure in one cannot contaminate the next. We keep
going on failure and report at the end—one bad metro should not cost you the other eight.

Everything between running the cells and uploading a byte is a gate, and each gate is here
because something got past its absence:

**The metro actually changed.** `METRO` and `STATE` are read back out of the executed namespace
and compared to what we asked for. A broken substitution now stops on metro one instead of
publishing nine copies of the same city.

**The validation actually passed.** `CHECKS` is read out of the namespace and every entry must
be `True`. We also refuse a run that produced *no* checks, or implausibly few—that means the
validation section did not run, and "no failures" from a suite that never executed is the most
dangerous green there is.

**The validation actually ran against this metro's data.** `DATASET` is confirmed to be the
scratch dataset. If it still says `a4i_food`, the override missed and the checks just graded
somebody's leftovers.

**Each dataframe is captured the moment it exists**, rather than read out of the namespace at
the end. A later cell that binds the same name to something else—an `int`, say—used to take a
table out of the snapshot silently.

In [ ]:
import io
import time
import traceback
import pandas as pd
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

# The student notebook runs 23 checks today. Demand most of them rather than an
# exact count, so adding a check does not break publishing, but deleting the
# whole section does.
MIN_CHECKS = 18

results = {}

for metro, (state, _fips) in METROS.items():
    print(f"\n{'=' * 64}\n{metro}\n{'=' * 64}")
    t0 = time.time()
    ns = {"__name__": "__main__"}
    captured = {}
    try:
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, metro)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

            # Grab each table as soon as it appears. Whatever a later cell does
            # to the name afterwards is then none of our business.
            for var in TABLES.values():
                val = ns.get(var)
                if isinstance(val, pd.DataFrame):
                    captured[var] = val

        # --- Gate 1: did the overrides actually take? ------------------------
        ran_as = ns.get("METRO")
        if ran_as != metro:
            raise RuntimeError(
                f"metro override failed: asked for {metro!r}, notebook ran as {ran_as!r}. "
                f"Nothing uploaded. Fix configure() before rerunning."
            )
        if ns.get("STATE") != state:
            raise RuntimeError(f"{metro} ran with STATE={ns.get('STATE')!r}, expected {state!r}")
        if ns.get("DATASET") != SCRATCH_DATASET:
            raise RuntimeError(
                f"{metro} validated against DATASET={ns.get('DATASET')!r}, not "
                f"{SCRATCH_DATASET!r}. Those checks graded the wrong tables."
            )

        # --- Gate 2: did the student notebook's own validation pass? ---------
        checks = ns.get("CHECKS")
        if not checks:
            raise RuntimeError(
                "the validation section produced no checks - it did not run. Refusing to "
                "publish something nobody checked."
            )
        if len(checks) < MIN_CHECKS:
            raise RuntimeError(
                f"only {len(checks)} checks ran, expected at least {MIN_CHECKS}. The "
                f"validation section is incomplete."
            )
        failed = [(name, detail) for name, passed, detail in checks if not passed]
        if failed:
            lines = "; ".join(f"{n} ({d})" for n, d in failed)
            raise RuntimeError(f"{len(failed)} of {len(checks)} checks FAILED: {lines}")
        print(f"  validation             {len(checks)} checks, all passed")

        # --- Gate 3: is every table actually here? ---------------------------
        missing = [f"{t} ({v})" for t, v in TABLES.items() if v not in captured]
        if missing:
            raise RuntimeError("never saw a dataframe for: " + ", ".join(missing))

        slug = metro.lower().replace(" ", "-")
        for table, var in TABLES.items():
            df = captured[var]
            df = df.drop(columns=[c for c in ("tract_geom",) if c in df.columns])
            if df.empty:
                raise RuntimeError(f"{table} is empty - refusing to publish it")
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            buf.seek(0)
            blob = bucket.blob(f"{PREFIX}/{slug}/{table}/data.parquet")
            blob.upload_from_file(buf, content_type="application/octet-stream")
            print(f"  {table:<22} {len(df):>7,} rows -> gs://{BUCKET}/{blob.name}")

        results[metro] = ("OK", time.time() - t0)
    except Exception as exc:                              # noqa: BLE001
        print(f"  FAILED: {exc}")
        traceback.print_exc()
        results[metro] = (f"FAILED: {exc}", time.time() - t0)

print(f"\n{'=' * 64}\nSUMMARY\n{'=' * 64}")
for metro, (status, secs) in results.items():
    print(f"  {metro:<16} {secs:>6.1f}s  {status}")

bad = [m for m, (s, _) in results.items() if s != "OK"]
if bad:
    print(f"\n{len(bad)} metro(s) did not publish: {', '.join(bad)}")
    print("Their previous snapshot, if any, is untouched - nothing partial was uploaded.")

## Verify the snapshot is loadable—and is actually the city it claims to be

Publishing is not the same as publishing something that works, and something that works is not
the same as something that is right. This cell reads every file back the way `load.sh` will,
then asks four questions of it:

1. **Is it there, and does it have rows?** The floor.
2. **Profile variety.** The one number that decides whether vector search can work at all. A
   corpus where every profile is identical would load cleanly and rank meaninglessly.
3. **Is this the right city?** Organizations must be in the expected state, the metro's own
   name must appear among its cities, and every census tract id must begin with that state's
   FIPS code. This is the check that was missing when nine metros published Chicago's data—the
   row counts were all plausible, because a wrong city still has a believable number of
   pantries in it.
4. **Are the metros actually different from each other?** We fingerprint each metro's recipient
   set. Two metros with an identical fingerprint means the override collapsed again.

`shelf_life` is deliberately excluded from 3 and 4: it is USDA FoodKeeper, identical everywhere.

It also prints the two numbers the README's metro table needs—real organization count and population-weighted no-vehicle rate—as a paste-ready markdown table, so those figures come from the published snapshot rather than from somebody's memory of an earlier run.

In [ ]:
import hashlib

problems = []
fingerprints = {}
readme = {m: {"orgs": None, "no_vehicle": None} for m in METROS}


def flag(metro, msg):
    problems.append(f"{metro}: {msg}")
    return "   <-- " + msg


for metro, (state, fips) in METROS.items():
    slug = metro.lower().replace(" ", "-")
    print(f"\n{metro}")
    for table in TABLES:
        path = f"{PREFIX}/{slug}/{table}/data.parquet"
        blob = bucket.blob(path)
        if not blob.exists():
            print(f"  {table:<20} MISSING  gs://{BUCKET}/{path}")
            problems.append(f"{metro}: {table} missing")
            continue

        df = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        note = ""

        if len(df) == 0:
            note += flag(metro, f"{table} is empty")

        if table == "recipients":
            variety = df["profile_text"].nunique() / max(len(df), 1)
            note += f"  variety {variety:.0%}"
            if variety < 0.9:
                note += flag(metro, f"profiles only {variety:.0%} distinct - regenerate")

            states = df["state"].astype(str).str.strip().str.upper()
            top_state = states.mode().iat[0] if len(states) else "?"
            note += f"  state {top_state}"
            if top_state != state:
                note += flag(metro, f"organizations are in {top_state}, expected {state}")

            cities = df["city"].astype(str).str.strip().str.upper()
            if metro.upper() not in set(cities):
                note += flag(metro, f"no organization has city == {metro.upper()!r} "
                                    f"(top city is {cities.mode().iat[0]!r})")

            fingerprints[metro] = hashlib.sha1(
                "".join(sorted(df["ein"].astype(str))).encode()).hexdigest()[:12]
            # README's metro table counts real organizations, not the corpus - the
            # nine planted cases are ours, not the city's.
            readme[metro]["orgs"] = int((df["profile_source"] == "generated").sum())

        if table == "tract_demographics":
            gid = df["geo_id"].astype(str)
            bad_len = int((gid.str.len() != 11).sum())
            in_state = float((gid.str[:2] == fips).mean()) if len(gid) else 0.0
            top_fips = gid.str[:2].mode().iat[0] if len(gid) else "??"
            note += f"  FIPS {top_fips} ({in_state:.0%})"
            # Population-weighted, not a mean of tract rates: a 40-person tract and a
            # 6,000-person tract should not count the same in a number we publish.
            w = df["total_pop"].fillna(0)
            nv = df["no_vehicle_rate"]
            ok_rows = nv.notna() & (w > 0)
            if ok_rows.any():
                readme[metro]["no_vehicle"] = float(
                    (nv[ok_rows] * w[ok_rows]).sum() / w[ok_rows].sum())
                note += f"  no-vehicle {readme[metro]['no_vehicle']:.0%}"
            if bad_len:
                note += flag(metro, f"{bad_len} tract ids are not 11 characters")
            # A metro bounding box is allowed to cross a state line - Camden sits
            # in the Philadelphia box, Newark in the New York one, and those tracts
            # belong there. What is not allowed is the majority being elsewhere.
            if top_fips != fips or in_state < 0.5:
                note += flag(metro, f"only {in_state:.0%} of tracts are in state FIPS "
                                    f"{fips} (majority is {top_fips}) - THIS IS THE WRONG CITY")

        print(f"  {table:<20} {len(df):>7,} rows{note}")

# --- Are any two metros the same data? -------------------------------------
print(f"\n{'=' * 64}\nRecipient fingerprints\n{'=' * 64}")
seen = {}
for metro, fp in fingerprints.items():
    dup = seen.get(fp)
    print(f"  {metro:<16} {fp}" + (f"   <-- IDENTICAL TO {dup}" if dup else ""))
    if dup:
        problems.append(f"{metro} and {dup} published identical recipient sets")
    seen[fp] = metro

# --- The two numbers the README's metro table wants ------------------------
print(f"\n{'=' * 64}\nREADME metro table - paste-ready\n{'=' * 64}")
print("| Metro | Organizations | Households with no vehicle |")
print("|---|---:|---:|")
for metro in METROS:
    r = readme[metro]
    orgs = f"{r['orgs']:,}" if r["orgs"] is not None else "—"
    nv   = f"{r['no_vehicle']:.0%}" if r["no_vehicle"] is not None else "—"
    print(f"| **{metro}** | {orgs} | {nv} |")
print("\n(Organizations = real IRS organizations, excluding the nine planted cases.")
print(" No-vehicle rate is population-weighted across the metro's census tracts.)")

print(f"\n{'=' * 64}")
if problems:
    print("Problems above. Do not announce this snapshot.\n")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All good. {len(METROS)} metros, {len(TABLES)} tables each, "
          f"every one in the state it claims and distinct from the others.")